In [2]:
import sys
sys.path.insert(0, "/tmp/modtest/src")
 
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stage3_metrics.decentralisation import all_metrics

In [6]:
def heuristic_pool(tag, addr, H):
    """Combine the tag and address columns into one pool label per block (Series)."""
    has_tag, has_addr = tag.notna(), addr.notna()
    if H == "H1":                                   # tag only
        return tag.where(has_tag, "unknown")
    if H == "H2":                                   # address only
        return addr.where(has_addr, "unknown")
    if H == "H3":                                   # union, tag-preferred on conflict
        return tag.where(has_tag, addr.where(has_addr, "unknown"))
    if H == "H4":                                   # intersection (both methods agree)
        agree = has_tag & has_addr & (tag == addr)
        return tag.where(agree, "unknown")
    if H == "H5":                                   # union, address-preferred on conflict
        return addr.where(has_addr, tag.where(has_tag, "unknown"))
    raise ValueError(H)

def apply_unknown_treatment(counts, treatment):
    counts = dict(counts)
    unknown = counts.pop("unknown", 0)
    if treatment == "U1":
        if unknown > 0:
            counts["unknown"] = unknown
    elif treatment == "U2":
        pass
    elif treatment == "U3":
        if counts and unknown > 0:
            counts[max(counts, key=counts.get)] += unknown
        elif unknown > 0:
            counts["unknown"] = unknown
    return counts

In [7]:
# ---- heuristic correctness: all five cases ----
tag = pd.Series(["A", "A", "A", None, None])
addr = pd.Series(["A", "B", None, "B", None])
expected = {
    "H1": ["A", "A", "A", "unknown", "unknown"],
    "H2": ["A", "B", "unknown", "B", "unknown"],
    "H3": ["A", "A", "A", "B", "unknown"],
    "H4": ["A", "unknown", "unknown", "unknown", "unknown"],
    "H5": ["A", "B", "A", "B", "unknown"],
}
print("=== heuristic correctness (cases: agree, disagree, tag-only, addr-only, neither) ===")
for H, exp in expected.items():
    got = list(heuristic_pool(tag, addr, H))
    status = "OK " if got == exp else "XX "
    print(f"{status}{H}: {got}")
    assert got == exp, (H, got, exp)
print("all five heuristics correct\n")

=== heuristic correctness (cases: agree, disagree, tag-only, addr-only, neither) ===
OK H1: ['A', 'A', 'A', 'unknown', 'unknown']
OK H2: ['A', 'B', 'unknown', 'B', 'unknown']
OK H3: ['A', 'A', 'A', 'B', 'unknown']
OK H4: ['A', 'unknown', 'unknown', 'unknown', 'unknown']
OK H5: ['A', 'B', 'A', 'B', 'unknown']
all five heuristics correct



In [8]:
# ---- full sweep on synthetic time data ----
def sweep_full(attributed, heuristics, treatments, period="M"):
    df = attributed.copy()
    df["window"] = pd.to_datetime(df["timestamp"]).dt.to_period(period).dt.to_timestamp()
    rows = []
    for H in heuristics:
        df["_pool"] = heuristic_pool(df["tag_pool"], df["addr_pool"], H)
        for window, g in df.groupby("window"):
            base = g["_pool"].value_counts().to_dict()
            for U in treatments:
                counts = apply_unknown_treatment(base, U)
                for name, value in all_metrics(counts).items():
                    rows.append({"window": window, "heuristic": H, "unknown": U,
                                 "metric": name, "value": value})
    return pd.DataFrame(rows)
 
rng = np.random.default_rng(3)
rows = []
pools = list("ABCDEFGH")
for year in range(2015, 2025):
    n_pools = min(2 + (year - 2015), len(pools))
    for _ in range(1200):
        ts = pd.Timestamp(f"{year}-06-01") + pd.Timedelta(days=int(rng.integers(-150, 150)))
        p = pools[rng.integers(n_pools)]
        # tag present ~70%, address present ~35% (sparser, like the real data)
        tg = p if rng.random() < 0.70 else None
        ad = p if rng.random() < 0.35 else None
        rows.append({"timestamp": ts, "tag_pool": tg, "addr_pool": ad})
attr = pd.DataFrame(rows)
 
H_ALL = ["H1", "H2", "H3", "H4", "H5"]
U_ALL = ["U1", "U2", "U3"]
sweep = sweep_full(attr, H_ALL, U_ALL)
print(f"sweep rows: {len(sweep):,} | {sweep['heuristic'].nunique()} heuristics x "
      f"{sweep['unknown'].nunique()} treatments x {sweep['metric'].nunique()} metrics")

sweep rows: 9,000 | 5 heuristics x 3 treatments x 6 metrics


In [9]:
# ---- axis decomposition (recent era) ----
def axis_band(metric, fix_col, fix_val, vary_col, year_from=2020):
    sub = sweep[(sweep["metric"] == metric) & (sweep[fix_col] == fix_val)]
    p = sub.pivot_table(index="window", columns=vary_col, values="value")
    spread = p.max(axis=1) - p.min(axis=1)
    return spread[spread.index.year >= year_from].mean()
 
print("\n=== which axis drives more sensitivity (mean band, 2020+) ===")
print(f"{'metric':9} {'heuristic-axis (varyH@U2)':>26} {'unknown-axis (varyU@H1)':>26}")
for m in ["nakamoto", "hhi", "shannon", "cr5"]:
    h = axis_band(m, "unknown", "U2", "heuristic")
    u = axis_band(m, "heuristic", "H1", "unknown")
    print(f"{m:9} {h:26.3f} {u:26.3f}")
 


=== which axis drives more sensitivity (mean band, 2020+) ===
metric     heuristic-axis (varyH@U2)    unknown-axis (varyU@H1)
nakamoto                       0.940                      1.620
hhi                            0.029                      0.106
shannon                        0.175                      0.419
cr5                            0.096                      0.090


In [16]:
# ---- full-envelope plot ----
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, metric in zip(axes.ravel(), ["nakamoto", "hhi", "gini", "shannon", "cr3", "cr5"]):
    sub = sweep[sweep["metric"] == metric]
    p = sub.pivot_table(index="window", columns=["heuristic", "unknown"], values="value")
    ax.fill_between(p.index, p.min(axis=1), p.max(axis=1), color="grey", alpha=0.2)
    base = sub[(sub["heuristic"] == "H1") & (sub["unknown"] == "U2")].set_index("window")["value"]
    ax.plot(base.index, base.values, color="#264653", lw=1.2, label="H1,U2 baseline")
    ax.set_title(metric)
axes.ravel()[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig("../figures/test_full_sweep_fig.png", dpi=110, bbox_inches="tight")
print("\nfull-grid envelope figure saved OK — sweep + decomposition work end-to-end")



full-grid envelope figure saved OK — sweep + decomposition work end-to-end
